# DsfTrackGenerator

The `Keypoints` class and `tracksFromPairwiseMatches` helper are internal GTSFM data structures for feature matching. They turn pairwise image matches into multi-view feature tracks using a disjoint-set forest.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/sfm/doc/DsfTrackGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import gtsam
import numpy as np

## Build the feature-matching inputs

GTSFM represents each image's detections with a `Keypoints` object containing an $N 	imes 2$ coordinate matrix. Pairwise matches map an `IndexPair` to rows of corresponding keypoint indices. Ordinary Python lists and dictionaries are converted to the internal `KeypointsVector` and `MatchIndicesMap` types by the wrapper.

In [3]:
keypoints = [
    gtsam.gtsfm.Keypoints(np.array([[10.0, 20.0], [30.0, 40.0]])),
    gtsam.gtsfm.Keypoints(
        np.array([[50.0, 60.0], [70.0, 80.0], [90.0, 100.0]])
    ),
    gtsam.gtsfm.Keypoints(np.array([[110.0, 120.0], [130.0, 140.0]])),
]
matches = {
    gtsam.IndexPair(0, 1): np.array([[0, 0], [1, 1]], dtype=np.int32),
    gtsam.IndexPair(1, 2): np.array([[2, 0], [1, 1]], dtype=np.int32),
}

tracks = gtsam.gtsfm.tracksFromPairwiseMatches(matches, keypoints)
assert len(tracks) == 3
print("track count:", len(tracks))
for track in tracks:
    print(track.indexVector(), "->", track.measurementMatrix().tolist())

track count: 3
[0 1] -> [[10.0, 20.0], [50.0, 60.0]]
[0 1 2] -> [[30.0, 40.0], [70.0, 80.0], [130.0, 140.0]]
[1 2] -> [[90.0, 100.0], [110.0, 120.0]]


## Inspect the generated tracks

`tracksFromPairwiseMatches()` merges pairwise correspondences with a disjoint-set forest and returns `SfmTrack2d` objects. For each track, `indexVector()` lists the participating cameras and `measurementMatrix()` contains the image coordinates in the same order.

In [4]:
assert all(track.hasUniqueCameras() for track in tracks)
assert tracks[1].numberMeasurements() == 3
np.testing.assert_allclose(tracks[1].indexVector(), [0, 1, 2])

## When to use it

This is internal feature-matching infrastructure used by GTSFM. It is useful when converting pairwise matcher output into tracks before triangulation or bundle adjustment. A valid track contains at most one keypoint per camera, so check `hasUniqueCameras()` before using the result downstream.

[`SfmTrack`](SfmTrack.ipynb) explains the returned track objects. The focused [`Keypoints`](Keypoints.ipynb) page gives additional coordinate-convention details.

## Source

[`DsfTrackGenerator.h`](../DsfTrackGenerator.h)


## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.